# Bab 7: Membangun Aplikasi Sembang
## Permulaan Cepat API Model Microsoft Foundry

Buku nota ini diadaptasi dari [Repositori Contoh Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang merangkumi buku nota yang mengakses perkhidmatan [Azure OpenAI](notebook-azure-openai.ipynb).

> **Nota:** Models GitHub akan diberhentikan pada akhir Julai 2026. Buku nota ini kini menyasarkan [Model Microsoft Foundry](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), yang menawarkan katalog model percubaaan percuma yang sama dan pengalaman SDK Inferens AI Azure.


# Gambaran Keseluruhan  
"Model bahasa besar adalah fungsi yang memetakan teks kepada teks. Diberikan rentetan teks input, model bahasa besar akan cuba meramalkan teks yang akan datang seterusnya"(1). Nota "quickstart" ini akan memperkenalkan pengguna kepada konsep LLM peringkat tinggi, keperluan pakej teras untuk memulakan dengan AML, pengenalan lembut kepada reka bentuk isyarat, dan beberapa contoh pendek bagi pelbagai kes penggunaan. 


## Jadual Kandungan  

[Gambaran Keseluruhan](#overview)  
[Cara menggunakan Perkhidmatan OpenAI](#how-to-use-openai-service)  
[1. Mewujudkan Perkhidmatan OpenAI anda](#1.-creating-your-openai-service)  
[2. Pemasangan](#2.-installation)    
[3. Kelayakan](#3.-credentials)  

[Kes Penggunaan](#use-cases)    
[1. Meringkaskan Teks](#1.-summarize-text)  
[2. Mengklasifikasikan Teks](#2.-classify-text)  
[3. Menjana Nama Produk Baru](#3.-generate-new-product-names)  
[4. Menyempurnakan Pengelasan](#4.fine-tune-a-classifier)  

[Rujukan](#references)


### Bina prompt pertama anda  
Latihan ringkas ini akan memberikan pengenalan asas untuk menyerahkan prompt kepada model dalam Microsoft Foundry Models untuk tugasan ringkas "meringkaskan".


**Langkah-langkah**:  
1. Pasang perpustakaan `azure-ai-inference` dalam persekitaran python anda, jika anda belum melakukannya.  
2. Muatkan perpustakaan pembantu standard dan tetapkan kelayakan Microsoft Foundry Models anda.  
3. Pilih model untuk tugasan anda  
4. Cipta prompt mudah untuk model  
5. Hantar permintaan anda ke API model!


### 1. Pasang `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Import perpustakaan pembantu dan instansikan kelayakan


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Mencari model yang sesuai  
Model seperti GPT-4o dan GPT-4o mini boleh memahami dan menjana bahasa semula jadi, dan tersedia dalam katalog Model Microsoft Foundry bersama model dari Meta, Mistral, Cohere, dan Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Reka Bentuk Prompt  

"Keajaiban model bahasa besar ialah dengan dilatih untuk meminimumkan ralat ramalan ini ke atas sejumlah besar teks, model-model ini akhirnya mempelajari konsep yang berguna untuk ramalan ini. Contohnya, mereka mempelajari konsep seperti"(1):

* bagaimana mengeja
* bagaimana tatabahasa berfungsi
* bagaimana untuk mengolah ayat semula
* bagaimana menjawab soalan
* bagaimana untuk mengadakan perbualan
* bagaimana menulis dalam pelbagai bahasa
* bagaimana untuk menulis kod
* dan sebagainya.

#### Cara mengawal model bahasa besar  
"Daripada semua input ke model bahasa besar, yang paling berpengaruh adalah teks prompt(1).

Model bahasa besar boleh diprompt untuk menghasilkan output dengan beberapa cara:

Arahan: Beritahu model apa yang anda mahu
Lengkapkan: Galakkan model untuk melengkapkan permulaan apa yang anda mahu
Demonstrasi: Tunjukkan kepada model apa yang anda mahu, sama ada:
Beberapa contoh dalam prompt
Beratus-ratus atau beribu-ribu contoh dalam set data latihan penalaan halus"



#### Terdapat tiga garis panduan asas untuk membuat prompt:

**Tunjuk dan beritahu**. Nyatakan dengan jelas apa yang anda mahu sama ada melalui arahan, contoh, atau gabungan kedua-duanya. Jika anda mahu model menyusun satu senarai item secara abjad atau mengklasifikasikan satu perenggan mengikut sentimen, tunjukkan kepadanya itu yang anda mahu.

**Sediakan data berkualiti**. Jika anda cuba membina pengklasifikasi atau mahu model mengikuti pola, pastikan terdapat cukup contoh. Pastikan anda menyemak semula contoh anda — model biasanya cukup bijak untuk mengesan kesilapan ejaan asas dan memberi anda respons, tetapi ia juga mungkin menganggap ini disengajakan dan ia boleh menjejaskan respons itu.

**Semak tetapan anda.** Tetapan temperature dan top_p mengawal sejauh mana deterministik model dalam menjana respons. Jika anda meminta respons di mana hanya ada satu jawapan yang betul, maka anda ingin menetapkannya lebih rendah. Jika anda mahu respons yang lebih pelbagai, maka anda boleh menetapkannya lebih tinggi. Kesilapan nombor satu yang dilakukan orang dengan tetapan ini adalah menganggap ia ialah kawalan "kepintaran" atau "kreativiti".


Sumber: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Hantar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Ringkaskan Teks  
#### Cabaran  
Ringkaskan teks dengan menambah 'tl;dr:' di akhir petikan teks. Perhatikan bagaimana model memahami cara melaksanakan beberapa tugas tanpa arahan tambahan. Anda boleh mencuba petunjuk yang lebih deskriptif daripada tl;dr untuk mengubah tingkah laku model dan menyesuaikan ringkasan yang anda terima(3).  

Kerja terkini telah menunjukkan peningkatan ketara dalam banyak tugas dan penanda aras NLP dengan pra-latihan pada korpus teks yang besar diikuti dengan penyempurnaan pada tugas tertentu. Walaupun biasanya tugas-agnostik dalam seni bina, kaedah ini masih memerlukan set data penyempurnaan khusus tugas yang mengandungi ribuan atau puluhan ribu contoh. Sebaliknya, manusia biasanya dapat melaksanakan tugas bahasa baru hanya dengan beberapa contoh atau arahan mudah - sesuatu yang sistem NLP semasa masih agak sukar lakukan. Di sini kami menunjukkan bahawa peningkatan skala model bahasa sangat meningkatkan prestasi tugas-agnostik dengan sedikit contoh, kadang-kadang mencapai tahap bersaing dengan pendekatan penyempurnaan terkini yang terbaik.  



Tl;dr  


# Latihan untuk beberapa kes penggunaan  
1. Rumuskan Teks  
2. Klasifikasikan Teks  
3. Jana Nama Produk Baharu


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasifikasikan Teks  
#### Cabaran  
Klasifikasikan item ke dalam kategori yang diberikan semasa masa inferens. Dalam contoh berikut, kami menyediakan kedua-dua kategori dan teks untuk diklasifikasikan dalam arahan (*playground_reference). 

Pertanyaan Pelanggan: Hai, salah satu kekunci pada papan kekunci komputer riba saya baru-baru ini rosak dan saya memerlukan pengganti:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Jana Nama Produk Baru
#### Cabaran
Cipta nama produk daripada kata contoh. Di sini kami sertakan dalam arahan maklumat tentang produk yang akan kami jana nama. Kami juga menyediakan contoh yang serupa untuk menunjukkan corak yang kami inginkan. Kami juga telah menetapkan nilai suhu tinggi untuk meningkatkan kebetulan dan respons yang lebih inovatif.

Penerangan produk: Pembuat milkshake di rumah
Kata benih: pantas, sihat, padat.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Penerangan produk: Sepasang kasut yang sesuai untuk saiz kaki mana-mana.
Kata benih: boleh disesuaikan, sesuai, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Rujukan  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Amalan terbaik untuk menyesuaikan model GPT bagi mengklasifikasikan teks](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Untuk Bantuan Lebih Lanjut  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Penyumbang
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
